In [ ]:
import sys
import subprocess

required = ["torch", "transformers", "datasets", "scikit-learn", "numpy"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])
print("Installed required packages.")


In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")


In [ ]:
model_name = "textattack/distilbert-base-uncased-MRPC"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

print(f"Loaded model: {model_name}")
print(f"Number of labels: {model.config.num_labels}")


In [ ]:
max_examples = 128
dataset = load_dataset("glue", "mrpc", split=f"validation[:{max_examples}]")

print("Dataset split: glue/mrpc validation")
print(f"Using first {len(dataset)} examples for fast evaluation")
print("Example row:")
print(dataset[0])


In [ ]:
pair_lengths = []
s1_lengths = []
s2_lengths = []

for row in dataset:
    s1_ids = tokenizer.encode(row["sentence1"], add_special_tokens=False)
    s2_ids = tokenizer.encode(row["sentence2"], add_special_tokens=False)
    s1_len = len(s1_ids)
    s2_len = len(s2_ids)
    pair_len = s1_len + s2_len
    s1_lengths.append(s1_len)
    s2_lengths.append(s2_len)
    pair_lengths.append(pair_len)

pair_lengths = np.array(pair_lengths)
s1_lengths = np.array(s1_lengths)
s2_lengths = np.array(s2_lengths)

q1, q2 = np.quantile(pair_lengths, [1/3, 2/3])
short_threshold = int(np.floor(q1))
medium_threshold = int(np.floor(q2))

bucket_names = []
for length in pair_lengths:
    if length <= short_threshold:
        bucket_names.append("short")
    elif length <= medium_threshold:
        bucket_names.append("medium")
    else:
        bucket_names.append("long")

print("Token-length statistics based on sentence1_tokens + sentence2_tokens (without special tokens):")
print(f"sentence1_len_min={int(s1_lengths.min())}, sentence1_len_mean={s1_lengths.mean():.2f}, sentence1_len_max={int(s1_lengths.max())}")
print(f"sentence2_len_min={int(s2_lengths.min())}, sentence2_len_mean={s2_lengths.mean():.2f}, sentence2_len_max={int(s2_lengths.max())}")
print(f"pair_len_min={int(pair_lengths.min())}, pair_len_mean={pair_lengths.mean():.2f}, pair_len_median={np.median(pair_lengths):.2f}, pair_len_max={int(pair_lengths.max())}")
print(f"bucket_threshold_short_le={short_threshold}")
print(f"bucket_threshold_medium_le={medium_threshold}")

for bucket in ["short", "medium", "long"]:
    count = sum(1 for b in bucket_names if b == bucket)
    print(f"bucket={bucket}, count={count}")


In [ ]:
batch_size = 32
labels = dataset["label"]
predictions = []
confidences = []
predicted_positive_prob = []

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(logits, dim=-1)

    predictions.extend(preds.cpu().tolist())
    confidences.extend(probs.max(dim=-1).values.cpu().tolist())
    predicted_positive_prob.extend(probs[:, 1].cpu().tolist())

print(f"Completed inference for {len(predictions)} examples.")


In [ ]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, support_binary = precision_recall_fscore_support(
    labels, predictions, average="binary", zero_division=0
)
cm = confusion_matrix(labels, predictions)
per_class_precision, per_class_recall, per_class_f1, per_class_support = precision_recall_fscore_support(
    labels, predictions, labels=[0, 1], average=None, zero_division=0
)

print("Overall evaluation metrics on fixed 128-example subset:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)

print("Per-class metrics:")
label_map = {0: "not_paraphrase", 1: "paraphrase"}
for idx, label_id in enumerate([0, 1]):
    print(
        f"class={label_id} ({label_map[label_id]}), "
        f"precision={per_class_precision[idx]:.4f}, "
        f"recall={per_class_recall[idx]:.4f}, "
        f"f1={per_class_f1[idx]:.4f}, "
        f"support={per_class_support[idx]}"
    )


In [ ]:
records = []
for i in range(len(dataset)):
    row = dataset[i]
    records.append({
        "index": i,
        "label": labels[i],
        "prediction": predictions[i],
        "confidence": confidences[i],
        "p_paraphrase": predicted_positive_prob[i],
        "sentence1_len": int(s1_lengths[i]),
        "sentence2_len": int(s2_lengths[i]),
        "pair_len": int(pair_lengths[i]),
        "bucket": bucket_names[i],
    })

print(f"Prepared {len(records)} evaluation records with token-length metadata.")


In [ ]:
print("Bucketed performance by token-length bucket:")
bucket_results = {}

for bucket in ["short", "medium", "long"]:
    subset = [r for r in records if r["bucket"] == bucket]
    y_true = [r["label"] for r in subset]
    y_pred = [r["prediction"] for r in subset]
    bucket_acc = accuracy_score(y_true, y_pred)
    bucket_precision, bucket_recall, bucket_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    bucket_cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    bucket_pair_lengths = [r["pair_len"] for r in subset]
    bucket_results[bucket] = {
        "count": len(subset),
        "accuracy": bucket_acc,
        "precision": bucket_precision,
        "recall": bucket_recall,
        "f1": bucket_f1,
        "confusion_matrix": bucket_cm.tolist(),
        "pair_len_min": int(min(bucket_pair_lengths)),
        "pair_len_mean": float(np.mean(bucket_pair_lengths)),
        "pair_len_max": int(max(bucket_pair_lengths)),
    }
    print(
        f"bucket={bucket}, count={len(subset)}, "
        f"pair_len_min={bucket_results[bucket]['pair_len_min']}, "
        f"pair_len_mean={bucket_results[bucket]['pair_len_mean']:.2f}, "
        f"pair_len_max={bucket_results[bucket]['pair_len_max']}, "
        f"accuracy={bucket_acc:.4f}, precision={bucket_precision:.4f}, recall={bucket_recall:.4f}, f1={bucket_f1:.4f}, "
        f"confusion_matrix={bucket_cm.tolist()}"
    )


In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation[:128]")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"confusion_matrix={cm.tolist()}")
print(f"support_not_paraphrase={int(per_class_support[0])}")
print(f"support_paraphrase={int(per_class_support[1])}")
print(f"pair_len_min={int(pair_lengths.min())}")
print(f"pair_len_mean={pair_lengths.mean():.2f}")
print(f"pair_len_median={np.median(pair_lengths):.2f}")
print(f"pair_len_max={int(pair_lengths.max())}")
print(f"short_threshold_le={short_threshold}")
print(f"medium_threshold_le={medium_threshold}")
for bucket in ["short", "medium", "long"]:
    br = bucket_results[bucket]
    print(f"bucket_{bucket}_count={br['count']}")
    print(f"bucket_{bucket}_accuracy={br['accuracy']:.4f}")
    print(f"bucket_{bucket}_precision={br['precision']:.4f}")
    print(f"bucket_{bucket}_recall={br['recall']:.4f}")
    print(f"bucket_{bucket}_f1={br['f1']:.4f}")
    print(f"bucket_{bucket}_confusion_matrix={br['confusion_matrix']}")
